# Sentinel: train, export and evaluate

Runs the whole pipeline on a free GPU (Colab or Kaggle) in about 30–45 minutes.

**Before you start:** Colab: Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Kaggle: Session options → Accelerator **GPU T4 x2** and Internet **on**, then Run All.

| Step | What it does | What to check |
|---|---|---|
| 1 | Install | No red errors |
| 2 | Build dataset | Every split has rows and an injection rate between 0 and 1 |
| 3 | Look at the data | Label 1 examples look like attacks, label 0 like normal text |
| 4 | Fine-tune | Validation F1 goes up across epochs |
| 5 | Export to ONNX | The three checks pass (the script stops if one fails) |
| 6 | Evaluate | Tables appear: copy them into the README |
| 7 | Tests | All green |
| 8 | Download / publish | You get `sentinel_artifacts.zip` |

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"

## 1. Install

In [ ]:
!git clone -q https://github.com/FernandoSuarezzz/sentinel.git
%cd sentinel
!pip install -q -e ".[train,dev]"

## 2. Build the dataset (public datasets + synthetic indirect injections, EN/ES)

In [ ]:
!python -m sentinel.data --out artifacts/data/dataset.parquet
assert _exit_code == 0, "data failed: read the error above; the next steps would only repeat it"

## 3. Look at the data
Never train on data you have not looked at. One example per source and label:

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 160)
df = pd.read_parquet("artifacts/data/dataset.parquet")
df.groupby(["source", "label"]).head(1)[["source", "label", "lang", "text"]]

## 4. Fine-tune (~10–20 min on a T4)

In [ ]:
!python -m sentinel.train --config configs/train.yaml
assert _exit_code == 0, "train failed: read the error above; the next steps would only repeat it"

## 5. Export to ONNX + int8 quantization, with automatic checks

In [ ]:
!python -m sentinel.export --config configs/train.yaml
assert _exit_code == 0, "export failed: read the error above; the next steps would only repeat it"

## 6. Evaluate the shipped model and compare with the baseline

In [ ]:
!python -m sentinel.evaluate --config configs/train.yaml
assert _exit_code == 0, "evaluate failed: read the error above; the next steps would only repeat it"
from IPython.display import Markdown

Markdown(open("artifacts/report/report.md").read())

## 7. Run the test suite

In [ ]:
!pytest -q

## 8. Download the results

In [ ]:
!cd artifacts && zip -qr ../sentinel_artifacts.zip onnx report && cd ..
!ls -lh sentinel_artifacts.zip
try:  # Colab: download straight away
    from google.colab import files
    files.download("sentinel_artifacts.zip")
except ImportError:  # Kaggle: the file appears in the Output panel on the right
    print("Download sentinel_artifacts.zip from the Output panel (folder sentinel).")

## 9. (Optional) Publish the model for the live demo

Needs a free Hugging Face account and a **write** token (huggingface.co → Settings → Access Tokens).
Change `HF_USER` to your Hugging Face username.

In [ ]:
from huggingface_hub import HfApi, notebook_login

notebook_login()

HF_USER = "FernandoSuarezzz"  # <- your Hugging Face username
repo_id = f"{HF_USER}/sentinel-prompt-injection"
api = HfApi()
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path="artifacts/onnx", repo_id=repo_id,
                  allow_patterns=["model.int8.onnx", "tokenizer.json",
                                  "sentinel_config.json", "export_report.json"])
print(f"https://huggingface.co/{repo_id}")